# Демо: asyncio

Прокликай Shift+Enter каждую ячейку и посмотри, как `asyncio` решает ту же задачу «делать несколько дел одновременно» в одном потоке: `async def` + `await`, `asyncio.run`, `asyncio.gather` для параллели, `asyncio.create_task`, `asyncio.to_thread` для блокирующих вызовов. В конце — три мини-задания.

**Зависимость для Jupyter.** Jupyter уже крутит свой event loop, поэтому `asyncio.run()` напрямую в ячейке падает с `RuntimeError: cannot be called from a running event loop`. Чинится пакетом `nest_asyncio` — он патчит event loop, разрешая вложенные `run`. Если пакет не установлен:

```
pip install nest_asyncio
```

В обычном `.py` скрипте этот хак не нужен — там один корневой `asyncio.run(main())` и проблем нет.

In [1]:
import nest_asyncio
nest_asyncio.apply()
print("nest_asyncio applied — теперь asyncio.run работает в Jupyter")


nest_asyncio applied — теперь asyncio.run работает в Jupyter


## Часть 1. Первая корутина — `async def` + `asyncio.run`

Функция, объявленная через `async def`, при вызове **не выполняется**. Она возвращает объект-корутину, которую надо запустить через event loop. Самый простой способ — `asyncio.run(coroutine)`.

In [2]:
import asyncio

async def hello(name):
    print(f"  привет, {name}")
    await asyncio.sleep(0.1)
    print(f"  пока, {name}")
    return f"<{name}>"

# Просто вызов НЕ выполняет тело — возвращает корутину
coro = hello("мир")
print("тип объекта:", type(coro).__name__)

# Чтобы запустить — нужен event loop
result = asyncio.run(coro)
print("результат:", result)


тип объекта: coroutine
  привет, мир
  пока, мир
результат: <мир>


Заметь: `print("тип объекта: ...")` отработал **до** того, как корутина выполнилась. `asyncio.run` запустил event loop, прокрутил корутину до конца и закрыл loop. После `asyncio.run` корутина уже отработала.

## Часть 2. `await` внутри корутины — последовательно

Если внутри `async def` мы `await`-им несколько корутин подряд — они выполняются последовательно. Каждая ждёт свою задержку. Общее время — сумма.

In [3]:
import asyncio
import time

async def fake_request(name, latency):
    print(f"  start  {name}")
    await asyncio.sleep(latency)
    print(f"  done   {name}")
    return f"<{name}>"

async def main():
    start = time.perf_counter()
    r1 = await fake_request("alpha", 0.2)
    r2 = await fake_request("beta",  0.3)
    r3 = await fake_request("gamma", 0.2)
    elapsed = time.perf_counter() - start
    print(f"\nрезультаты: {[r1, r2, r3]}")
    print(f"время:      {elapsed:.2f}s — сумма (0.2 + 0.3 + 0.2 = 0.7)")

asyncio.run(main())


  start  alpha


  done   alpha
  start  beta


  done   beta
  start  gamma


  done   gamma

результаты: ['<alpha>', '<beta>', '<gamma>']
время:      0.70s — сумма (0.2 + 0.3 + 0.2 = 0.7)


## Часть 3. `asyncio.gather` — параллельный запуск

Чтобы запустить корутины параллельно, оборачиваем их в `asyncio.gather(...)`. Все три задачи стартуют почти одновременно, общее время — самая медленная (не сумма).

In [4]:
import asyncio
import time

async def fake_request(name, latency):
    print(f"  start  {name}")
    await asyncio.sleep(latency)
    print(f"  done   {name}")
    return f"<{name}>"

async def main():
    start = time.perf_counter()
    results = await asyncio.gather(
        fake_request("alpha", 0.2),
        fake_request("beta",  0.3),
        fake_request("gamma", 0.2),
    )
    elapsed = time.perf_counter() - start
    print(f"\nрезультаты: {results}")
    print(f"время:      {elapsed:.2f}s — близко к 0.3 (max), не 0.7 (sum)")

asyncio.run(main())


  start  alpha
  start  beta
  start  gamma


  done   alpha
  done   gamma
  done   beta

результаты: ['<alpha>', '<beta>', '<gamma>']
время:      0.30s — близко к 0.3 (max), не 0.7 (sum)


Это и есть главная сила asyncio: тысячи I/O-задач параллельно в одном потоке, без процессов и без потоков. Типичный сценарий в NLP — «отправить 50 запросов в LLM» через `gather` сокращает 50 запросов с десятков секунд до 1-2 секунд.

## Часть 4. `asyncio.create_task` — фоновая задача

`create_task(coro)` ставит корутину в event loop как **фоновую задачу**: она стартует немедленно, не дожидаясь `await`. Удобно, когда нужно запустить много задач, делать что-то параллельно с ними и потом забрать результат.

In [5]:
import asyncio
import time

async def background_work(name, latency):
    print(f"  фоновая {name} стартует")
    await asyncio.sleep(latency)
    print(f"  фоновая {name} закончила")
    return name

async def main():
    start = time.perf_counter()
    # Стартуем три фоновых задачи — они уже идут
    t1 = asyncio.create_task(background_work("task-1", 0.3))
    t2 = asyncio.create_task(background_work("task-2", 0.2))
    t3 = asyncio.create_task(background_work("task-3", 0.4))
    print("  основной код продолжает выполняться, пока задачи в фоне...")
    await asyncio.sleep(0.05)
    print("  основной код подождал немного, теперь забирает результаты")
    # Дожидаемся всех
    results = [await t1, await t2, await t3]
    elapsed = time.perf_counter() - start
    print(f"\nрезультаты: {results}")
    print(f"время:      {elapsed:.2f}s — близко к 0.4 (самая медленная)")

asyncio.run(main())


  основной код продолжает выполняться, пока задачи в фоне...
  фоновая task-1 стартует
  фоновая task-2 стартует
  фоновая task-3 стартует
  основной код подождал немного, теперь забирает результаты


  фоновая task-2 закончила
  фоновая task-1 закончила


  фоновая task-3 закончила

результаты: ['task-1', 'task-2', 'task-3']
время:      0.40s — близко к 0.4 (самая медленная)


## Часть 5. `asyncio.to_thread` — выносим блокирующий код в поток

Иногда нужно из `async def` позвать обычную блокирующую функцию (нет async-версии библиотеки или legacy). Если позвать напрямую — она заморозит event loop на всё время выполнения. `asyncio.to_thread` выносит вызов в отдельный поток, event loop остаётся свободным.

In [6]:
import asyncio
import time

def blocking_compute(name, seconds):
    print(f"  start  {name}")
    time.sleep(seconds)   # БЛОКИРУЮЩИЙ — нельзя напрямую в async def
    print(f"  done   {name}")
    return f"<{name}>"

async def main():
    start = time.perf_counter()
    # Три блокирующих вызова, но через to_thread они идут параллельно
    results = await asyncio.gather(
        asyncio.to_thread(blocking_compute, "alpha", 0.3),
        asyncio.to_thread(blocking_compute, "beta",  0.2),
        asyncio.to_thread(blocking_compute, "gamma", 0.4),
    )
    elapsed = time.perf_counter() - start
    print(f"\nрезультаты: {results}")
    print(f"время:      {elapsed:.2f}s — близко к 0.4 (max), не 0.9 (sum)")

asyncio.run(main())


  start  alpha  start  beta

  start  gamma


  done   beta
  done   alpha


  done   gamma

результаты: ['<alpha>', '<beta>', '<gamma>']
время:      0.41s — близко к 0.4 (max), не 0.9 (sum)


Что произойдёт без `to_thread` (если позвать `time.sleep` напрямую в корутине)? Event loop замёрзнет на 0.3+0.2+0.4 = 0.9 с — другие задачи будут ждать. На реальном backend это означает: один медленный блокирующий вызов и тысячи клиентов не получают ответа.

## Часть 6. Sequential vs gather — сравнение бенчмарков

Один и тот же набор задач, два способа запуска. Покажет картину сразу:

In [7]:
import asyncio
import time

async def fake_io(idx):
    await asyncio.sleep(0.1)
    return idx

N = 10

async def sequential():
    start = time.perf_counter()
    results = []
    for i in range(N):
        results.append(await fake_io(i))
    return time.perf_counter() - start, results

async def parallel():
    start = time.perf_counter()
    results = await asyncio.gather(*(fake_io(i) for i in range(N)))
    return time.perf_counter() - start, results

async def main():
    seq_time, seq_res = await sequential()
    par_time, par_res = await parallel()
    print(f"sequential: {seq_time:.2f}s — {N} задач × 0.1с подряд")
    print(f"gather:     {par_time:.2f}s — все параллельно")
    print(f"speedup:    {seq_time / par_time:.1f}x")
    print(f"совпали:    {seq_res == par_res}")

asyncio.run(main())


sequential: 1.02s — 10 задач × 0.1с подряд
gather:     0.10s — все параллельно
speedup:    9.9x
совпали:    True


## Часть 7. Таймаут на корутину — `asyncio.wait_for`

Реальные сетевые вызовы могут зависнуть. `asyncio.wait_for(coro, timeout)` ставит дедлайн: если корутина не успела — поднимется `asyncio.TimeoutError`.

In [8]:
import asyncio

async def slow_request(name, latency):
    await asyncio.sleep(latency)
    return f"<{name}>"

async def main():
    # 1. Успевает в дедлайн
    try:
        r = await asyncio.wait_for(slow_request("fast", 0.1), timeout=0.5)
        print("fast ответил:", r)
    except asyncio.TimeoutError:
        print("fast — TimeoutError")

    # 2. Не успевает
    try:
        r = await asyncio.wait_for(slow_request("slow", 0.5), timeout=0.1)
        print("slow ответил:", r)
    except asyncio.TimeoutError:
        print("slow — TimeoutError, не дождались")

asyncio.run(main())


fast ответил: <fast>


slow — TimeoutError, не дождались


## Мини-задания

Три коротких упражнения. Подсказок к именам и API нет — вспомни сам.

**Задание 1.** Напиши `async def fetch(url, latency)` — печатает 'fetch url', `await`-ит `asyncio.sleep(latency)`, возвращает `url`. В `main()` запусти 5 таких вызовов параллельно через `asyncio.gather`, замерь время и сравни с суммой задержек.

**Задание 2.** Напиши `async def fetch_many(urls)`, которая принимает список URL'ов и возвращает список результатов через `asyncio.gather`. Внутри используй list-comprehension `[fetch(u, 0.1) for u in urls]`.

**Задание 3.** Что напечатает код ниже? Сначала угадай (порядок выводов), потом запусти.

In [9]:
# Задание 1
# import asyncio, time
# async def fetch(url, latency):
#     ...
# async def main():
#     # 5 параллельных вызовов
# asyncio.run(main())


In [10]:
# Задание 2
# import asyncio
# async def fetch(url, latency):
#     ...
# async def fetch_many(urls):
#     ...
# urls = ['a', 'b', 'c', 'd']
# asyncio.run(fetch_many(urls))


In [11]:
# Задание 3 — угадай порядок выводов:
import asyncio

async def task(name, delay):
    print(f"  enter  {name}")
    await asyncio.sleep(delay)
    print(f"  exit   {name}")
    return name

async def main():
    results = await asyncio.gather(
        task("slow",   0.3),
        task("fast",   0.1),
        task("medium", 0.2),
    )
    print("results:", results)

asyncio.run(main())


  enter  slow
  enter  fast
  enter  medium


  exit   fast


  exit   medium


  exit   slow
results: ['slow', 'fast', 'medium']
